In [ ]:
!pip install -q --no-cache-dir -e .[dev]

## 0. Environment Setup

Install the package in editable mode with development dependencies so the notebook can import the local code and run the training and evaluation scripts.

# Model Training and Evaluation

This notebook turns the sample training workflow into a reusable template for a NanoDet-based experiment.

Update the placeholder paths, class names, and run name before executing the commands.

In [ ]:
from pathlib import Path
import subprocess
import sys
import yaml

PROJECT_ROOT = Path.cwd()
RUN_NAME = "your_run_name"
CONFIG_PATH = PROJECT_ROOT / "config" / f"{RUN_NAME}.yaml"
CHECKPOINT_PATH = PROJECT_ROOT / "runs" / RUN_NAME / "model_best" / "nanodet_model_best.pth"
SEED = 42

TRAIN_IMAGES = "<path/to/train/images>"
TRAIN_ANN = "<path/to/train/annotations.json>"
VAL_IMAGES = "<path/to/val/images>"
VAL_ANN = "<path/to/val/annotations.json>"
TEST_IMAGES = "<path/to/test/images>"
TEST_ANN = "<path/to/test/annotations.json>"
CP_CAL_IMAGES = "<path/to/calibration/images>"
CP_CAL_ANN = "<path/to/calibration/annotations.json>"
CP_TEST_IMAGES = "<path/to/conformal-test/images>"
CP_TEST_ANN = "<path/to/conformal-test/annotations.json>"

NUM_CLASSES = 4
CLASS_NAMES = ["class_1", "class_2", "class_3", "class_4"]

def run_command(command):
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")

In [ ]:
def make_dataset(img_path, ann_path, *, augment=False):
    pipeline = {
        "use_torchvision_v2": False,
        "normalize": [[103.53, 116.28, 123.675], [57.375, 57.12, 58.395]],
    }
    if augment:
        pipeline.update({
            "perspective": 0.0,
            "scale": [0.6, 1.4],
            "stretch": [[0.8, 1.2], [0.8, 1.2]],
            "rotation": 0,
            "shear": 0,
            "translate": 0.2,
            "flip": 0.5,
            "brightness": 0.2,
            "contrast": [0.6, 1.4],
            "saturation": [0.5, 1.2],
        })
    return {
        "name": "CocoDataset",
        "img_path": img_path,
        "ann_path": ann_path,
        "input_size": [640, 640],
        "keep_ratio": False,
        "pipeline": pipeline,
    }

config = {
    "save_dir": str(PROJECT_ROOT / "runs" / RUN_NAME),
    "model": {
        "weight_averager": {
            "name": "ExpMovingAverager",
            "decay": 0.9998,
        },
        "arch": {
            "name": "NanoDetPlus",
            "detach_epoch": 10,
            "backbone": {
                "name": "ShuffleNetV2",
                "model_size": "1.5x",
                "out_stages": [2, 3, 4],
                "activation": "LeakyReLU",
            },
            "fpn": {
                "name": "GhostPAN",
                "in_channels": [176, 352, 704],
                "out_channels": 128,
                "kernel_size": 5,
                "num_extra_level": 1,
                "use_depthwise": True,
                "activation": "LeakyReLU",
            },
            "head": {
                "name": "NanoDetPlusHead",
                "num_classes": NUM_CLASSES,
                "input_channel": 128,
                "feat_channels": 128,
                "stacked_convs": 2,
                "kernel_size": 5,
                "strides": [8, 16, 32, 64],
                "activation": "LeakyReLU",
                "reg_max": 7,
                "norm_cfg": {"type": "BN"},
                "loss": {
                    "loss_qfl": {
                        "name": "QualityFocalLoss",
                        "use_sigmoid": True,
                        "beta": 2.0,
                        "loss_weight": 1.0,
                    },
                    "loss_dfl": {
                        "name": "DistributionFocalLoss",
                        "loss_weight": 0.25,
                    },
                    "loss_bbox": {
                        "name": "GIoULoss",
                        "loss_weight": 2.0,
                    },
                },
            },
        },
    },
    "data": {
        "train": make_dataset(TRAIN_IMAGES, TRAIN_ANN, augment=True),
        "val": make_dataset(VAL_IMAGES, VAL_ANN),
        "test": make_dataset(TEST_IMAGES, TEST_ANN),
        "cp_cal": make_dataset(CP_CAL_IMAGES, CP_CAL_ANN),
        "cp_test": make_dataset(CP_TEST_IMAGES, CP_TEST_ANN),
    },
    "device": {
        "gpu_ids": [0],
        "workers_per_gpu": 4,
        "batchsize_per_gpu": 16,
        "precision": 32,
    },
    "schedule": {
        "optimizer": {
            "name": "AdamW",
            "lr": 0.001,
            "weight_decay": 0.05,
        },
        "resume": True,
        "warmup": {
            "name": "linear",
            "steps": 1000,
            "ratio": 0.0001,
        },
        "total_epochs": 100,
        "lr_schedule": {
            "name": "MultiStepLR",
            "milestones": [70, 90],
            "gamma": 0.1,
        },
        "val_intervals": 10,
    },
    "grad_clip": 35,
    "evaluator": {
        "name": "CocoDetectionEvaluator",
        "save_key": "val/mAP",
    },
    "log": {
        "interval": 100,
    },
    "class_names": CLASS_NAMES,
}

print("Configuration template assembled.")

## Training

Save the generated configuration and run the training script with a chosen seed.

In [ ]:
with open(CONFIG_PATH, "w") as file:
    yaml.safe_dump(config, file, sort_keys=False)

print(f"Wrote configuration to {CONFIG_PATH}")

train_command = [
    sys.executable,
    "tools/train.py",
    str(CONFIG_PATH),
    "--seed",
    str(SEED),
]

print("Training command ready:")
print(" ".join(train_command))
# run_command(train_command)

## Evaluation

Evaluate the trained checkpoint on the validation split and then generate calibration/test scores for conformal analysis.

In [ ]:
evaluation_commands = {
    "val": [
        sys.executable,
        "tools/test.py",
        "--config",
        str(CONFIG_PATH),
        "--model",
        str(CHECKPOINT_PATH),
        "--task",
        "val",
    ],
    "cp_cal": [
        sys.executable,
        "tools/test.py",
        "--config",
        str(CONFIG_PATH),
        "--model",
        str(CHECKPOINT_PATH),
        "--task",
        "cp_cal",
    ],
    "cp_test": [
        sys.executable,
        "tools/test.py",
        "--config",
        str(CONFIG_PATH),
        "--model",
        str(CHECKPOINT_PATH),
        "--task",
        "cp_test",
    ],
    "full_scores_cp_cal": [
        sys.executable,
        "tools/get_full_scores.py",
        "--config",
        str(CONFIG_PATH),
        "--model",
        str(CHECKPOINT_PATH),
        "--output",
        str(PROJECT_ROOT / "runs" / RUN_NAME / "cp_cal_scores.json"),
        "--task",
        "cp_cal",
    ],
    "full_scores_cp_test": [
        sys.executable,
        "tools/get_full_scores.py",
        "--config",
        str(CONFIG_PATH),
        "--model",
        str(CHECKPOINT_PATH),
        "--output",
        str(PROJECT_ROOT / "runs" / RUN_NAME / "cp_test_scores.json"),
        "--task",
        "cp_test",
    ],
}

for name, command in evaluation_commands.items():
    print(f"{name}: {' '.join(command)}")
# run_command(evaluation_commands['val'])
# run_command(evaluation_commands['cp_cal'])
# run_command(evaluation_commands['cp_test'])
# run_command(evaluation_commands['full_scores_cp_cal'])
# run_command(evaluation_commands['full_scores_cp_test'])

## Notes

- Replace the placeholder paths with the paths for your dataset and run directory.
- Adjust `NUM_CLASSES` and `CLASS_NAMES` to match the experiment.
- Use the generated calibration and test score files in the conformal risk control notebook.